# Pump Sensor Anomaly Detection
## Single-file Pipeline: EDA → Preprocessing → Model Comparison → Auto-Tuning → LLM → Gradio

**실행 환경**: Kaggle Notebook (T4 GPU 권장)  
**데이터셋**: `pump-sensor-data` — 52 sensors, ~220K rows  
**실행 방법**: Run All (Shift+F5) — 전체 자동 실행, 외부 파일 의존 없음

| Section | 내용 |
|---------|------|
| 0 | 설치 + Config |
| 1 | 데이터 로드 |
| 2 | 깊은 EDA (7종 시각화) |
| 3 | VIF 다중공선성 제거 + 피처 확정 |
| 4 | 전처리 + Sliding Window |
| 5 | 모델 정의 (5종) |
| 6 | 전체 학습 + 자동 최선 모델 선정 |
| 7 | Optuna 자동 튜닝 (선정 모델만) |
| 8 | 최종 재학습 + 평가 |
| 9 | LLM 리포트 + Gradio UI |

---
**Kaggle 빌드:** 시각화 제거·축약. 전체 그림은 `notebooks/clean/` 로컬 실행.



---
## Section 0. Install & Config

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  INSTALL & IMPORTS
# ─────────────────────────────────────────────────────────────────
import subprocess, sys
import os, time, warnings, pickle, copy, random

pkgs = ['statsmodels', 'optuna', 'gradio', 'openai']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    f1_score, fbeta_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.model_selection import train_test_split

import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import gradio as gr
import openai

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─────────────────────────────────────────────────────────────────
#  CONFIG
# [FLAG] TUNING_MODE
#  True  : Optuna 튜닝 단계 — 속도 최우선 (benchmark=True)
#  False : 최종 A/B 평가 단계 — 재현성 최우선 (deterministic=True)
# ─────────────────────────────────────────────────────────────────
TUNING_MODE = True

DATA_PATH = None
base_path = '/kaggle/input'
for root, dirs, files in os.walk(base_path):
    if 'sensor.csv' in files:
        DATA_PATH = os.path.join(root, 'sensor.csv')
        break

if DATA_PATH is None:
    raise FileNotFoundError('sensor.csv를 찾을 수 없습니다. 데이터셋이 추가되었는지 확인하세요.')

OUTPUT_DIR = '/kaggle/working/figures'
MODEL_DIR  = '/kaggle/working/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

# ── 전처리 파라미터
WINDOW_SIZE    = 50
STEP_SIZE      = 10
SCALER_TYPE    = 'minmax'
THRESHOLD_PCT  = 95

# ★ All-sensors 컷오프 무효화
MISSING_THRESH = 0.50
VIF_THRESH     = 9999.0
CORR_THRESH    = 1.0

# ── 기본 학습 파라미터
DEFAULT_HIDDEN  = 64
DEFAULT_LATENT  = 32
DEFAULT_LAYERS  = 2
DEFAULT_DROPOUT = 0.1
DEFAULT_LR      = 1e-3
DEFAULT_BATCH   = 64
DEFAULT_EPOCHS  = 100
OPTUNA_TRIALS   = 500

# ── 시드 고정
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    if TUNING_MODE:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark     = True   # 속도 최우선
    else:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False  # 재현성 최우선

print(f'Mode   : {"TUNING (속도 최우선)" if TUNING_MODE else "EVALUATION (재현성 최우선)"}')
print(f'Device : {device}')
print(f'Data   : {DATA_PATH}')
print('Config OK.')


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  검증 (Validation) 셀
# ─────────────────────────────────────────────────────────────────
import os

# 1. 파일 경로 확인
if DATA_PATH and os.path.exists(DATA_PATH):
    print(f"✅ 파일 탐색 성공: {DATA_PATH}")
    
    # 2. 데이터 로드 테스트
    try:
        df_check = pd.read_csv(DATA_PATH, nrows=5) # 5줄만 로드
        print(f"✅ 데이터 읽기 성공! Shape (샘플): {df_check.shape}")
        
        # 3. 필수 컬럼 존재 확인 (sensor_00 ~ sensor_51, machine_status)
        cols = df_check.columns.tolist()
        sensor_count = len([c for c in cols if 'sensor' in c])
        has_status = 'machine_status' in cols
        
        print(f"📊 센서 컬럼 개수: {sensor_count}개")
        print(f"📊 레이블(machine_status) 존재: {'YES' if has_status else 'NO'}")
        
        # 4. 데이터 샘플 출력
        print("\n=== 데이터 상위 3행 샘플 ===")
        display(df_check.head(3))
        
    except Exception as e:
        print(f"❌ 데이터 읽기 실패: {e}")
else:
    print(f"❌ 파일을 찾을 수 없습니다. 경로를 확인하세요: {DATA_PATH}")

# 5. 디바이스 및 출력 경로 재확인
print(f"\n💻 실행 디바이스: {device}")
print(f"📂 모델 저장 경로: {MODEL_DIR}")

---
## Section 1. Data Load

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = df_raw.columns.str.strip()

# 타임스탬프 컬럼 자동 감지
ts_col = next((c for c in df_raw.columns if 'time' in c.lower()), None)
if ts_col:
    df_raw.rename(columns={ts_col: 'timestamp'}, inplace=True)
    df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
    df_raw = df_raw.set_index('timestamp').sort_index()

sensor_cols = [c for c in df_raw.columns if c.startswith('sensor')]

print(f'Shape         : {df_raw.shape}')
print(f'Sensor cols   : {len(sensor_cols)}')
print(f'Date range    : {df_raw.index.min()} ~ {df_raw.index.max()}')
print(f'\nmachine_status counts:')
print(df_raw['machine_status'].value_counts())
df_raw.head(3)

---
## Section 2. Deep EDA — omitted (Kaggle no-viz)

원본 `notebooks/clean/` 에는 7종 시각화가 있습니다. 본 빌드는 Kaggle에서 학습·튜닝만 수행합니다. `valid_sensors` / `df_vis` 는 아래 셀에서 clean과 동일 로직으로만 정의합니다.


In [ ]:
# Section 2 시각화 생략 — 결측 기준만 동일 (clean 노트북과 동일)
missing = df_raw[sensor_cols].isnull().mean().sort_values(ascending=False)
drop_missing = missing[missing > MISSING_THRESH].index.tolist()
valid_sensors = [c for c in sensor_cols if c not in drop_missing]
df_vis = df_raw[valid_sensors + ['machine_status']].copy()
print(f'Dropped by missing: {drop_missing}')
print(f'Remaining sensors : {len(valid_sensors)}')


---
## Section 3. Feature Selection — VIF Multicollinearity Removal

`VIF (Variance Inflation Factor)` 해석:
- VIF = 1: 완전 독립
- VIF 1~5: 허용 수준
- VIF 5~10: 주의
- **VIF > 10: 제거** — 해당 피처가 다른 피처들의 선형결합으로 거의 설명됨

알고리즘: 가장 높은 VIF 피처부터 하나씩 제거 후 재계산 반복 (Backward Elimination)

In [ ]:
def compute_vif(df_feat):
    X = df_feat.dropna().astype(float)
    return pd.DataFrame({
        'feature': X.columns,
        'VIF':     [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }).sort_values('VIF', ascending=False).reset_index(drop=True)


def iterative_vif_drop(df_feat, threshold, verbose=True):
    remaining = list(df_feat.columns)
    log = []
    step = 0
    while True:
        step += 1
        vif = compute_vif(df_feat[remaining])
        top_vif   = vif.iloc[0]['VIF']
        top_feat  = vif.iloc[0]['feature']
        if top_vif <= threshold:
            if verbose: print(f'  [step {step}] Done — all VIF <= {threshold}')
            break
        if verbose: print(f'  [step {step}] Drop {top_feat:<15s}  VIF={top_vif:.2f}')
        log.append({'feature': top_feat, 'vif': round(top_vif,2), 'step': step})
        remaining.remove(top_feat)
    return remaining, pd.DataFrame(log)


# VIF 계산용 샘플 (속도 최적화)
vif_sample = df_vis[valid_sensors].dropna().sample(
    min(8000, len(df_vis)), random_state=SEED)

vif_before = compute_vif(vif_sample)
print('=== VIF Before Elimination (top 15) ===')
print(vif_before.head(15).to_string(index=False))

In [ ]:
print('\n=== Iterative VIF Elimination ===')
selected_by_vif, vif_log = iterative_vif_drop(vif_sample, VIF_THRESH)

# 제로 분산 제거
zero_var = [c for c in selected_by_vif if df_vis[c].std() < 1e-6]
FINAL_FEATURES = [c for c in selected_by_vif if c not in zero_var]
N_FEATURES = len(FINAL_FEATURES)

vif_after = compute_vif(vif_sample[FINAL_FEATURES])

print(f'\n  Original   : {len(valid_sensors)} sensors')
print(f'  Dropped (VIF)    : {[c for c in valid_sensors if c not in selected_by_vif]}')
print(f'  Dropped (zero-var): {zero_var}')
print(f'  FINAL : {N_FEATURES} features')
print(f'\nFinal features: {FINAL_FEATURES}')




---
## Section 4. Preprocessing — Normalization + Sliding Window + Split

In [ ]:
df_proc = df_raw[FINAL_FEATURES + ['machine_status']].copy()
df_proc = df_proc.reset_index(drop=True)

label_map = {'NORMAL': 0, 'BROKEN': 1, 'RECOVERING': 1}
df_proc['label'] = df_proc['machine_status'].map(label_map).fillna(0).astype(int)

# --- 시간 순서 기준 행 단위 train / val / test (랜덤 stratify 없음) ---
# 뒤 20~30%만 자르면 이 데이터에서 test에 이상 라벨이 없을 수 있어,
# train 55% / val 15% / test 30% 로 분할해 test에 RECOVERING 구간이 포함되도록 함.
n = len(df_proc)
SPLIT_TRAIN_FRAC, SPLIT_VAL_FRAC = 0.55, 0.15
i_tr = int(n * SPLIT_TRAIN_FRAC)
i_val = int(n * (SPLIT_TRAIN_FRAC + SPLIT_VAL_FRAC))

train_df = df_proc.iloc[:i_tr].copy()
val_df = df_proc.iloc[i_tr:i_val].copy()
test_df = df_proc.iloc[i_val:].copy()

def _impute_feat(df_part):
    df_part = df_part.copy()
    # 실시간/스트리밍: 미래 시점 참조 금지 → bfill 제거 (과거 ffill + 잔여는 0)
    df_part[FINAL_FEATURES] = df_part[FINAL_FEATURES].ffill().fillna(0.0)
    return df_part

train_df = _impute_feat(train_df)
val_df = _impute_feat(val_df)
test_df = _impute_feat(test_df)

print(f'[Split] n={n} | train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')
print(f'[Split] anomaly %  train {train_df["label"].mean()*100:.2f}% | val {val_df["label"].mean()*100:.2f}% | test {test_df["label"].mean()*100:.2f}%')

normal_mask_tr = train_df['label'] == 0
if SCALER_TYPE == 'minmax':
    scaler = MinMaxScaler()
else:
    scaler = StandardScaler()

scaler.fit(train_df.loc[normal_mask_tr, FINAL_FEATURES])
pickle.dump(scaler, open(f'{MODEL_DIR}/scaler.pkl', 'wb'))
pickle.dump(FINAL_FEATURES, open(f'{MODEL_DIR}/final_features.pkl', 'wb'))

def _safe_scaled(X):
    return np.nan_to_num(
        scaler.transform(X).astype(np.float32),
        nan=0.0, posinf=0.0, neginf=0.0,
    )

train_scaled = _safe_scaled(train_df[FINAL_FEATURES])
val_scaled = _safe_scaled(val_df[FINAL_FEATURES])
test_scaled = _safe_scaled(test_df[FINAL_FEATURES])

labels_train = train_df['label'].values
labels_val = val_df['label'].values
labels_test = test_df['label'].values

_all = np.vstack([train_scaled, val_scaled, test_scaled])
print(f'Scaler        : {SCALER_TYPE}')
print(f'Scaled range  : [{_all.min():.3f}, {_all.max():.3f}]')
print(f'Train / val / test shapes: {train_scaled.shape}, {val_scaled.shape}, {test_scaled.shape}')


In [ ]:
def make_windows(data, labels, win, step):
    """
    Sliding window 생성.
    window 내 이상 레이블이 하나라도 있으면 해당 window = 이상(1)
    Returns: X (N, win, feat), y (N,)
    """
    X, y = [], []
    for s in range(0, len(data) - win, step):
        X.append(data[s:s+win])
        y.append(int(labels[s:s+win].max()))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


X_train_all, y_train_all = make_windows(train_scaled, labels_train, WINDOW_SIZE, STEP_SIZE)
X_val_all, y_val_all = make_windows(val_scaled, labels_val, WINDOW_SIZE, STEP_SIZE)
X_test, y_test = make_windows(test_scaled, labels_test, WINDOW_SIZE, STEP_SIZE)

normal_idx = np.where(y_train_all == 0)[0]
X_train = X_train_all[normal_idx]

X_train_t = torch.FloatTensor(X_train).to(device)
X_val_t = torch.FloatTensor(X_val_all).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)

train_loader_default = DataLoader(
    TensorDataset(X_train_t, X_train_t),
    batch_size=DEFAULT_BATCH, shuffle=True,
    pin_memory=False, num_workers=0
)

print(f'X_train (normal) : {X_train_t.shape}  — Autoencoder 학습용')
print(f'X_val (all)      : {X_val_t.shape}')
print(f'X_test (all)     : {X_test_t.shape}')
print(f'y_val anomaly    : {y_val_all.mean()*100:.2f}%  |  y_test anomaly: {y_test.mean()*100:.2f}%')




---
## Section 5. Model Definitions (5종)

In [ ]:
# ── Model A: LSTM Autoencoder ──────────────────────────────────────
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_feat, hidden=64, latent=32, layers=2, dropout=0.1, **kw):
        super().__init__()
        self.model_name = 'LSTM-AE'
        drop = dropout if layers > 1 else 0
        self.enc   = nn.LSTM(n_feat, hidden, layers, batch_first=True, dropout=drop)
        self.fc_e  = nn.Linear(hidden, latent)
        self.fc_d  = nn.Linear(latent, hidden)
        self.dec   = nn.LSTM(hidden, hidden, layers, batch_first=True, dropout=drop)
        self.out   = nn.Linear(hidden, n_feat)

    def forward(self, x):
        _, (h, _) = self.enc(x)
        z   = self.fc_e(h[-1])
        di  = self.fc_d(z).unsqueeze(1).repeat(1, x.size(1), 1)
        dec, _ = self.dec(di)
        return self.out(dec)


# ── Model B: CNN-1D Autoencoder ───────────────────────────────────
class CNN1DAutoencoder(nn.Module):
    def __init__(self, n_feat, hidden=64, latent=32, dropout=0.1, **kw):
        super().__init__()
        self.model_name = 'CNN1D-AE'
        self.enc = nn.Sequential(
            nn.Conv1d(n_feat, hidden, 7, padding=3), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden*2, 5, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc_e = nn.Linear(hidden*2, latent)
        self.fc_d = nn.Linear(latent, hidden*2)
        self.dec = nn.Sequential(
            nn.ConvTranspose1d(hidden*2, hidden, 5, padding=2), nn.ReLU(), nn.Dropout(dropout),
            nn.ConvTranspose1d(hidden, n_feat, 7, padding=3)
        )

    def forward(self, x):
        W = x.size(1)
        z  = self.fc_e(self.enc(x.permute(0,2,1)).squeeze(-1))
        di = self.fc_d(z).unsqueeze(-1).repeat(1,1,W)
        return self.dec(di).permute(0,2,1)


# ── Model C: Transformer Autoencoder ─────────────────────────────
class TransformerAutoencoder(nn.Module):
    def __init__(self, n_feat, hidden=64, latent=32, nhead=4, layers=2, dropout=0.1, **kw):
        super().__init__()
        self.model_name = 'Transformer-AE'
        # nhead must divide hidden
        nhead = max(h for h in [1,2,4,8] if hidden % h == 0 and h <= nhead)
        self.proj = nn.Linear(n_feat, hidden)
        enc_l = nn.TransformerEncoderLayer(hidden, nhead, hidden*4, dropout, batch_first=True)
        self.tenc = nn.TransformerEncoder(enc_l, layers)
        self.fc_e = nn.Linear(hidden, latent)
        self.fc_d = nn.Linear(latent, hidden)
        dec_l = nn.TransformerDecoderLayer(hidden, nhead, hidden*4, dropout, batch_first=True)
        self.tdec = nn.TransformerDecoder(dec_l, layers)
        self.out  = nn.Linear(hidden, n_feat)

    def forward(self, x):
        p = self.proj(x)
        m = self.tenc(p)
        z = self.fc_e(m.mean(1))
        d = self.fc_d(z).unsqueeze(1).repeat(1, x.size(1), 1)
        return self.out(self.tdec(d, m))


print('Models A (LSTM-AE), B (CNN1D-AE), C (Transformer-AE) defined.')
print('Classical: D (Isolation Forest), E (One-Class SVM) — trained in Section 6.')

---
## Section 6. Train All Models → Auto-Select Best

**흐름**: 5개 모델 학습 → **test** 비교표(참고) → 딥러닝 3종은 **Val F2(β=2, recall 가중)** 으로만 `BEST_MODEL_NAME` 선정(test 미사용) → Section 7 Optuna(목표 Val F2)과 동일 기준

In [ ]:
# ── 공통 유틸리티 ──────────────────────────────────────────────────
OPTUNA_TRAIN_EPOCHS = 30  # objective() 루프·Cosine T_max·최종 재학습과 동일해야 함

def train_autoencoder(model, loader, epochs, patience=10, verbose=True,
                      optimizer=None, scheduler=None):
    """EarlyStopping + AMP. optimizer=None 이면 AdamW(DEFAULT_LR). scheduler는 epoch마다 step."""
    model.train()
    criterion  = nn.MSELoss()
    if optimizer is None:
        optimizer = torch.optim.AdamW(model.parameters(), lr=DEFAULT_LR, weight_decay=1e-5)
    scaler_amp = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    best_loss, no_imp, history = float('inf'), 0, []

    for epoch in range(1, epochs + 1):
        ep_loss = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            ep_loss += loss.item()
        if scheduler is not None:
            scheduler.step()
        avg = ep_loss / len(loader)
        history.append(avg)
        if avg < best_loss: best_loss, no_imp = avg, 0
        else: no_imp += 1
        if no_imp >= patience:
            if verbose: print(f'    EarlyStop @ {epoch}  best={best_loss:.6f}')
            break
        if verbose and epoch % 10 == 0:
            print(f'    Epoch {epoch:3d}  loss={avg:.6f}')
    return history


def get_errors(model, X_tensor, batch=512):
    """배치 단위 재구성 오차 계산 (AMP 적용)"""
    model.eval()
    errs = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch):
            b = X_tensor[i:i + batch]
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                recon = model(b)
            errs.extend(((b - recon) ** 2).mean(dim=(1, 2)).cpu().numpy())
    return np.array(errs)


def evaluate(errors, y_true, pct=THRESHOLD_PCT):
    errors = np.nan_to_num(errors, nan=0.0, posinf=0.0, neginf=0.0)
    """임계값 설정 + 분류 지표 계산"""
    thr    = np.percentile(errors, pct)
    y_pred = (errors > thr).astype(int)
    return dict(
        threshold = thr,
        y_pred    = y_pred,
        errors    = errors,
        f1        = f1_score(y_true, y_pred, zero_division=0),
        f2        = fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        precision = precision_score(y_true, y_pred, zero_division=0),
        recall    = recall_score(y_true, y_pred, zero_division=0),
        roc_auc   = roc_auc_score(y_true, errors),
        pr_auc    = average_precision_score(y_true, errors),
    )


print('Utilities ready.')


In [ ]:
results     = {}      # 모든 결과 저장
model_objs  = {}      # 딥러닝 모델 객체 저장 (Optuna 후 재사용)
DEEP_MODEL_CLASSES = {
    'LSTM-AE':        LSTMAutoencoder,
    'CNN1D-AE':       CNN1DAutoencoder,
    'Transformer-AE': TransformerAutoencoder,
}
plot_colors = ['#6366F1','#10B981','#F59E0B','#EF4444','#8B5CF6']

# ── 딥러닝 3종 학습 ────────────────────────────────────────────────
for i, (mname, ModelCls) in enumerate(DEEP_MODEL_CLASSES.items()):
    print(f'\n[{i+1}/5] Training {mname} ...')
    t0    = time.time()
    model = ModelCls(N_FEATURES, hidden=DEFAULT_HIDDEN, latent=DEFAULT_LATENT,
                     layers=DEFAULT_LAYERS, dropout=DEFAULT_DROPOUT, nhead=4).to(device)
    hist  = train_autoencoder(model, train_loader_default, DEFAULT_EPOCHS, patience=10)
    errs_val = get_errors(model, X_val_t)
    met_val  = evaluate(errs_val, y_val_all)
    errs  = get_errors(model, X_test_t)
    met   = evaluate(errs, y_test)
    met.update({
        'history': hist, 'train_time': round(time.time()-t0,1),
        'val_f1': met_val['f1'], 'val_f2': met_val['f2'],
        'val_pr_auc': met_val['pr_auc'], 'val_roc_auc': met_val['roc_auc'],
    })
    results[mname]    = met
    model_objs[mname] = model
    torch.save(model.state_dict(), f'{MODEL_DIR}/{mname}_default.pt')
    print(f'    Val F2={met["val_f2"]:.4f}  Val F1={met["val_f1"]:.4f}  |  Test F1={met["f1"]:.4f}  ({met["train_time"]}s)')

# ── 전통 ML 2종 ────────────────────────────────────────────────────
Xtr_flat = X_train_t.cpu().numpy().reshape(len(X_train_t), -1)
Xte_flat = X_test_t.cpu().numpy().reshape(len(X_test_t), -1)

print('\n[4/5] Training Isolation Forest ...')
t0 = time.time()
iforest = IsolationForest(n_estimators=200, contamination='auto', random_state=SEED, n_jobs=-1)
iforest.fit(Xtr_flat)
sc_if = -iforest.score_samples(Xte_flat)
met_if = evaluate(sc_if, y_test)
met_if['train_time'] = round(time.time()-t0,1)
results['IForest'] = met_if
print(f'    F1={met_if["f1"]:.4f}  ({met_if["train_time"]}s)')

print('\n[5/5] Training One-Class SVM ...')
t0 = time.time()
idx_s = np.random.choice(len(Xtr_flat), min(5000, len(Xtr_flat)), replace=False)
ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(Xtr_flat[idx_s])
sc_sv = -ocsvm.score_samples(Xte_flat)
met_sv = evaluate(sc_sv, y_test)
met_sv['train_time'] = round(time.time()-t0,1)
results['OCSVM'] = met_sv
print(f'    F1={met_sv["f1"]:.4f}  ({met_sv["train_time"]}s)')

In [ ]:
# ── 성능 비교 (표만; Kaggle no-viz) ───────────────────────────────────────────────
model_names  = list(results.keys())
metrics_keys = ['f1','f2','precision','recall','roc_auc','pr_auc']

comp_df = pd.DataFrame(
    {k: {m: round(results[m][k],4) for m in model_names} for k in metrics_keys}
).T
print('\n=== Model Comparison (metrics on hold-out TEST — reference only; selection uses VAL) ===')
print(comp_df.to_string())




In [ ]:
# ══════════════════════════════════════════════════════════════════
#  자동 모델 선정 — 딥러닝 3종만, 기준: Val F2 (β=2, recall 가중 / Optuna와 동일, test 미관여)
#  IForest / OCSVM 은 선정 후보 아님(비교 기준선)
# ══════════════════════════════════════════════════════════════════
BEST_MODEL_NAME = max(
    DEEP_MODEL_CLASSES.keys(),
    key=lambda m: results[m]['val_f2'],
)
BEST_MODEL_CLS  = DEEP_MODEL_CLASSES[BEST_MODEL_NAME]

print('=' * 55)
print(f'  AUTO-SELECTED BEST MODEL: {BEST_MODEL_NAME}  (by Val F2, β=2)')
print(f'  Val F2 (선정 기준) : {results[BEST_MODEL_NAME]["val_f2"]:.4f}')
print(f'  Val F1 (참고)      : {results[BEST_MODEL_NAME]["val_f1"]:.4f}')
print(f'  Val PR-AUC         : {results[BEST_MODEL_NAME]["val_pr_auc"]:.4f}')
print(f'  Test F2 (참고)     : {results[BEST_MODEL_NAME]["f2"]:.4f}')
print(f'  Test F1 (참고)     : {results[BEST_MODEL_NAME]["f1"]:.4f}')
print(f'  Test ROC-AUC       : {results[BEST_MODEL_NAME]["roc_auc"]:.4f}')
print(f'  Test PR-AUC        : {results[BEST_MODEL_NAME]["pr_auc"]:.4f}')
print(f'  → Section 7에서 이 모델만 Optuna 튜닝 (Val F2)')
print('=' * 55)

---
## Section 7. Optuna Auto-Tuning (선정 모델 자동 적용)

`BEST_MODEL_NAME`이 자동으로 설정된 상태에서 바로 실행됩니다.
탐색 공간: hidden/latent 크기, layer 수, dropout, lr, batch_size, optimizer, threshold_pct

In [ ]:
# --- Window Cache (train/val/test 각각 동일 분할 기준) ---
print('Building window cache (upfront)...')
WINDOW_CACHE = {}
for _ws in [30, 50, 100, 150]:
    Xt, yt = make_windows(train_scaled, labels_train, _ws, STEP_SIZE)
    Xv, yv = make_windows(val_scaled, labels_val, _ws, STEP_SIZE)
    Xte, yte = make_windows(test_scaled, labels_test, _ws, STEP_SIZE)
    _ni = np.where(yt == 0)[0]
    WINDOW_CACHE[_ws] = {
        'X_train': torch.FloatTensor(Xt[_ni]),
        'X_val': torch.FloatTensor(Xv),
        'y_val': yv,
        'X_test': torch.FloatTensor(Xte),
        'y_test': yte,
    }
print(f'Cache ready: {list(WINDOW_CACHE.keys())}')

print(f'Optuna target: {BEST_MODEL_NAME}  |  n_trials={OPTUNA_TRIALS}')
best_trial_f2 = 0.0


def objective(trial):
    hidden = trial.suggest_categorical('hidden', [32, 64, 128, 256])
    latent = trial.suggest_categorical('latent', [8, 16, 32, 64])
    # CNN1D-AE는 클래스 내부 Conv 깊이가 고정이라 n_layers 미사용(LSTM/Transformer만 탐색)
    if BEST_MODEL_NAME == 'CNN1D-AE':
        n_layers = 2
    else:
        n_layers = trial.suggest_int('n_layers', 1, 3)
    dropout = trial.suggest_float('dropout', 0.0, 0.4, step=0.05)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch = trial.suggest_categorical('batch', [32, 64, 128, 256])
    opt_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'RMSprop'])
    thr_pct = trial.suggest_categorical('threshold_pct', [90, 93, 95, 97, 99])
    window_size = trial.suggest_categorical('window_size', [30, 50, 100, 150])

    cache = WINDOW_CACHE[window_size]
    X_tr_normal = cache['X_train'].to(device)
    X_va_t = cache['X_val'].to(device)
    y_va_arr = cache['y_val']

    loader = DataLoader(
        TensorDataset(X_tr_normal, X_tr_normal),
        batch_size=batch, shuffle=True,
        pin_memory=False, num_workers=0
    )

    model = BEST_MODEL_CLS(
        N_FEATURES, hidden=hidden, latent=latent,
        layers=n_layers, dropout=dropout, nhead=4
    ).to(device)

    opt_map = {'Adam': torch.optim.Adam,
               'AdamW': torch.optim.AdamW,
               'RMSprop': torch.optim.RMSprop}
    optimizer = opt_map[opt_name](model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=OPTUNA_TRAIN_EPOCHS, eta_min=1e-6)
    criterion = nn.MSELoss()
    scaler_amp = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    model.train()
    for epoch in range(OPTUNA_TRAIN_EPOCHS):
        ep_loss = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            ep_loss += loss.item()
        scheduler.step()
        trial.report(ep_loss / len(loader), epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    errs = get_errors(model, X_va_t)
    ev = evaluate(errs, y_va_arr, thr_pct)
    f2 = ev['f2']

    global best_trial_f2
    if f2 > best_trial_f2:
        best_trial_f2 = f2
        torch.save(model.state_dict(), f'{MODEL_DIR}/{BEST_MODEL_NAME}_optuna_best.pt')
        print(f'  [Trial {trial.number}] New best Val F2={f2:.4f} → saved')

    return f2


study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, catch=(ValueError,))

BEST_PARAMS = study.best_params
BEST_PARAMS.setdefault('n_layers', 2)
print(f'\nOptuna Best Val F2 : {study.best_value:.4f}')
print(f'Sec.6 default Val F2 (same model, pre-Optuna): {results[BEST_MODEL_NAME]["val_f2"]:.4f}')
print(f'Sec.6 Test F2/F1 (참고): {results[BEST_MODEL_NAME]["f2"]:.4f} / {results[BEST_MODEL_NAME]["f1"]:.4f}')
print(f'Best Params        : {BEST_PARAMS}')


---
## Section 8. Final Retrain with Best Params + Full Evaluation

In [ ]:
p = BEST_PARAMS
cache_final = WINDOW_CACHE[p['window_size']]
X_tr_fin = cache_final['X_train'].to(device)
X_te_fin = cache_final['X_test'].to(device)
y_te_fin = cache_final['y_test']

print(f'Retraining {BEST_MODEL_NAME} with tuned params: {p}')
print(f'  → window_size={p["window_size"]} (train/val/test 텐서는 WINDOW_CACHE와 동일)')

final_model = BEST_MODEL_CLS(
    N_FEATURES,
    hidden  = p['hidden'],
    latent  = p['latent'],
    layers  = p['n_layers'],
    dropout = p['dropout'],
    nhead   = 4
).to(device)

opt_map = {'Adam': torch.optim.Adam, 'AdamW': torch.optim.AdamW, 'RMSprop': torch.optim.RMSprop}
final_optimizer = opt_map[p['optimizer']](final_model.parameters(),
                                           lr=p['lr'], weight_decay=p['weight_decay'])
final_loader = DataLoader(TensorDataset(X_tr_fin, X_tr_fin),
                           batch_size=p['batch'], shuffle=True)
sched_fin = torch.optim.lr_scheduler.CosineAnnealingLR(
    final_optimizer, T_max=OPTUNA_TRAIN_EPOCHS, eta_min=1e-6)

final_hist = train_autoencoder(
    final_model, final_loader, OPTUNA_TRAIN_EPOCHS, patience=999, verbose=True,
    optimizer=final_optimizer, scheduler=sched_fin)
final_errors  = get_errors(final_model, X_te_fin)
final_metrics = evaluate(final_errors, y_te_fin, p['threshold_pct'])

torch.save(final_model.state_dict(), f'{MODEL_DIR}/{BEST_MODEL_NAME}_tuned.pt')

print(f'\n=== Final Tuned Model: {BEST_MODEL_NAME} ===')
print(f'  F2        : {final_metrics["f2"]:.4f}  F1: {final_metrics["f1"]:.4f}  (Sec.6 test F2/F1: {results[BEST_MODEL_NAME]["f2"]:.4f}/{results[BEST_MODEL_NAME]["f1"]:.4f}, val F2/F1: {results[BEST_MODEL_NAME]["val_f2"]:.4f}/{results[BEST_MODEL_NAME]["val_f1"]:.4f})')
print(f'  Precision : {final_metrics["precision"]:.4f}')
print(f'  Recall    : {final_metrics["recall"]:.4f}')
print(f'  ROC-AUC   : {final_metrics["roc_auc"]:.4f}')
print(f'  PR-AUC    : {final_metrics["pr_auc"]:.4f}')
print(f'  Threshold : {final_metrics["threshold"]:.6f}')

In [ ]:
# Default vs Tuned 전체 비교표
compare_rows = []
for mname in list(DEEP_MODEL_CLASSES.keys()) + ['IForest','OCSVM']:
    r = results[mname]
    compare_rows.append({
        'Model': mname,
        'F1': round(r['f1'],4), 'F2': round(r['f2'],4), 'Prec': round(r['precision'],4),
        'Recall': round(r['recall'],4), 'ROC-AUC': round(r['roc_auc'],4),
        'PR-AUC': round(r['pr_auc'],4), 'Time(s)': r['train_time'], 'Type': 'Default'
    })
compare_rows.append({
    'Model': f'{BEST_MODEL_NAME} (Tuned)', 'Type': 'Tuned',
    'F1': round(final_metrics['f1'],4), 'F2': round(final_metrics['f2'],4), 'Prec': round(final_metrics['precision'],4),
    'Recall': round(final_metrics['recall'],4), 'ROC-AUC': round(final_metrics['roc_auc'],4),
    'PR-AUC': round(final_metrics['pr_auc'],4), 'Time(s)': '-'
})
final_comparison = pd.DataFrame(compare_rows).set_index('Model')
print('\n=== Final Comparison Table ===')
print(final_comparison.to_string())

---
## Section 9. LLM Diagnostic Report + Gradio UI

**Stage 2 (로컬 제출물):** `rag_pipeline.py` + Chroma + **Gemini**(또는 Vertex) 기반 RAG 진단 리포트로 고도화되어 있습니다.  
**본 노트북(Kaggle):** 시연·재현을 위해 OpenAI `gpt-4o-mini`를 **선택적으로** 호출합니다(API Secret 없으면 규칙 기반 데모).

In [ ]:
# Kaggle: OpenAI gpt-4o-mini 시연. 로컬 Stage 2는 rag_pipeline + Gemini RAG로 동일 목적(본 섹션 상단 설명).
def extract_segments(errors, y_pred):
    segs, in_seg = [], False
    for i in range(len(y_pred)):
        if y_pred[i]==1 and not in_seg:
            start=i; in_seg=True
        elif y_pred[i]==0 and in_seg:
            segs.append({'start':start,'end':i,'duration':i-start,
                          'max_e':float(errors[start:i].max()),
                          'mean_e':float(errors[start:i].mean())})
            in_seg=False
    return segs


def make_report(metrics, model_name, n_feat, api_key=None):
    segs = extract_segments(metrics['errors'], metrics['y_pred'])
    thr  = metrics['threshold']

    if not api_key or not api_key.strip():
        lines = [f'# Anomaly Report — {model_name} (Rule-based Demo)',
                 f'F1={metrics["f1"]:.4f}  ROC-AUC={metrics["roc_auc"]:.4f}',
                 f'Detected segments: {len(segs)}', '']
        for i,s in enumerate(segs):
            sev = 'HIGH' if s['max_e']>thr*3 else 'MEDIUM' if s['max_e']>thr*1.5 else 'LOW'
            lines += [f'Segment {i+1}: idx {s["start"]}~{s["end"]} ({s["duration"]} windows)',
                      f'  Severity={sev}  max_err={s["max_e"]:.5f}', '']
        lines += ['Recommendation:',
                  '  Immediate: Check pump seals, bearings, and pressure relief valves.',
                  '  Short-term: Increase sensor sampling rate for anomalous channels.',
                  '  Long-term: Schedule preventive maintenance within 48h if HIGH detected.']
        return '\n'.join(lines)

    try:
        client = openai.OpenAI(api_key=api_key)
        seg_txt = '\n'.join(
            f'  Seg{i+1}: idx {s["start"]}~{s["end"]} dur={s["duration"]}w max_err={s["max_e"]:.5f}'
            for i,s in enumerate(segs[:8])
        )
        prompt = (
            f'Industrial pump anomaly detection.\n'
            f'Model: {model_name} ({n_feat} sensors)\n'
            f'F1={metrics["f1"]:.4f}  ROC-AUC={metrics["roc_auc"]:.4f}\n'
            f'Detected segments:\n{seg_txt}\n\n'
            f'Provide: (1) cause analysis per segment, (2) severity rating, '
            f'(3) recommended maintenance actions, (4) overall health summary.'
        )
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role':'system','content':'You are an industrial IoT anomaly detection expert.'},
                      {'role':'user','content':prompt}],
            max_tokens=1000, temperature=0.4
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f'[API Error] {e}\n\n' + make_report(metrics, model_name, n_feat)


# 테스트 출력 (API 키 없이)
print(make_report(final_metrics, BEST_MODEL_NAME + ' (Tuned)', N_FEATURES))

In [ ]:
# ── Gradio 앱 ─────────────────────────────────────────────────────
GRADIO_MODELS = {
    f'{BEST_MODEL_NAME} (Tuned)': final_model,
    **{k: model_objs[k] for k in model_objs},
}

def gradio_run(model_choice, thr_pct, api_key):
    sel_model = GRADIO_MODELS.get(model_choice, final_model)
    if model_choice == f'{BEST_MODEL_NAME} (Tuned)':
        _c = WINDOW_CACHE[BEST_PARAMS['window_size']]
        X_eval, y_eval = _c['X_test'].to(device), _c['y_test']
    else:
        X_eval, y_eval = X_test_t, y_test
    errs = get_errors(sel_model, X_eval)
    met  = evaluate(errs, y_eval, int(thr_pct))

    # Kaggle no-viz: Gradio 플롯 생략
    fig = plt.figure(figsize=(6, 1))
    fig.text(0.5, 0.5, 'Plot omitted (Kaggle no-viz)', ha='center', va='center')

    stats_text = (
        f'Model: {model_choice}\n'
        f'F1={met["f1"]:.4f}  Precision={met["precision"]:.4f}  Recall={met["recall"]:.4f}\n'
        f'ROC-AUC={met["roc_auc"]:.4f}  PR-AUC={met["pr_auc"]:.4f}\n'
        f'Threshold={met["threshold"]:.6f}  Detected={int(met["y_pred"].sum())}'
    )
    report = make_report(met, model_choice, N_FEATURES,
                         api_key=api_key.strip() or None)
    return fig, stats_text, report


model_choices = list(GRADIO_MODELS.keys())

with gr.Blocks(title='Pump Sensor Anomaly Detection', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        f'# Pump Sensor Anomaly Detection\n'
        f'**Best model (auto-selected + tuned): {BEST_MODEL_NAME}**  '
        f'F1={final_metrics["f1"]:.4f}  ROC-AUC={final_metrics["roc_auc"]:.4f}'
    )
    with gr.Row():
        with gr.Column(scale=1):
            model_dd  = gr.Dropdown(model_choices,
                                    value=f'{BEST_MODEL_NAME} (Tuned)',
                                    label='Model')
            thr_sl    = gr.Slider(85, 99, value=int(BEST_PARAMS.get('threshold_pct',95)),
                                  step=1, label='Threshold Percentile')
            api_box   = gr.Textbox(label='OpenAI API Key — Kaggle 시연용 (로컬은 Gemini+RAG)',
                                   type='password',
                                   placeholder='sk-... (없으면 규칙 기반 리포트 생성)')
            run_btn   = gr.Button('Run Analysis', variant='primary', size='lg')
            stats_out = gr.Textbox(label='Metrics Summary', lines=5)
        with gr.Column(scale=2):
            plot_out = gr.Plot(label='Detection Result')
    report_out = gr.Textbox(label='LLM Diagnostic Report', lines=22, max_lines=30)
    run_btn.click(
        fn=gradio_run,
        inputs=[model_dd, thr_sl, api_box],
        outputs=[plot_out, stats_out, report_out]
    )

print('Gradio app ready.')

In [ ]:
demo.launch(share=True, debug=False)
# share=True → 공개 URL 생성 (발표 시연용)